In [ ]:
import numpy as np
import pandas as pd

In [ ]:
train = pd.read_csv("/kaggle/input/competitions/industrial-surface-defect-detection/public/train.csv")

In [ ]:
train.head()

## Q1 — RLE encoding (8×8 grid)
Purple cells were read from the figure. Column-major numbering makes 24 (bottom of col 3) and 25 (top of col 4) neighbours, so they join one run.
Answer: 12 2 23 4 44 2 47 1

In [ ]:
# rle_m = train['mask_rle'].notna()
# print(rle_m)
# # train['mask_rle'][0].split(" ")

In [ ]:
img = "/kaggle/input/competitions/industrial-surface-defect-detection/public/train/0ZUU8LH57Y.jpg"

## Q2 — RLE decoding and pixel mass (image 0ZUU8LH57Y)
rle_decode turns "start length" pairs into a 128×800 mask: subtract 1 from each start (RLE is 1-based), set each run to 1 in a flat array, then reshape column by column (order="F"). Empty RLE → all zeros.
Per class: a 250, b 0, c 1232, d 5032. Answer: 6514

In [ ]:
def rle_decode(rle, h=128, w=800):
    # Empty mask_rle means "no defect" → return an all-zero mask
    if pd.isna(rle) or str(rle).strip() == "":
        return np.zeros((h, w), dtype=np.uint8)

    nums = np.array(str(rle).split(), dtype=int)   # "2 3 9 2" → [2, 3, 9, 2]
    starts = nums[0::2] - 1                         # [2, 9] → minus 1 → [1, 8]
    lengths = nums[1::2]                            # [3, 2]

    flat = np.zeros(h * w, dtype=np.uint8)          # one long line of zeros
    for s, l in zip(starts, lengths):
        flat[s:s + l] = 1                           # put 1s along each run

    return flat.reshape((h, w), order="F")          # fold into the grid, column by column

In [ ]:
#testing
print(rle_decode("2 3 9 2", h=4, w=4))

In [ ]:
# Q2: total defect pixels for image 0ZUU8LH57Y
rows = train[train["image_id"] == "0ZUU8LH57Y"]    # keep only this image's 4 rows

total = 0
for _, row in rows.iterrows():                      # go through the rows one by one
    mask = rle_decode(row["mask_rle"])              # RLE text → 128×800 mask
    count = int(mask.sum())                         # number of 1s = defect pixels
    print(row["defect_type"], mask.shape, count)
    total += count

print("Total defect pixels:", total)

## Q3 — Images per defect class
A filled mask_rle means the image has that defect. Counting unique image_ids per defect_type:
c 4100 · d 1707 · a 1081 · b 313.
Answer: defect_c (4100), defect_b (313). Strong imbalance: c is ~13× more common than b.

In [ ]:
# Q3: number of images containing each defect type
train["has_defect"] = train["mask_rle"].notna()
print(train[["image_id", "defect_type", "has_defect"]].head(4))

In [ ]:
positive = train[train["has_defect"]]      # keep only rows where a defect is present

counts = (positive.groupby("defect_type")["image_id"]
                  .nunique()
                  .sort_values(ascending=False))
print(counts)

print("Most common:", counts.index[0], counts.iloc[0])
print("Rarest    :", counts.index[-1], counts.iloc[-1])

## Q4 — Defect co-occurrence (4×4)
presence = one row per image, 1/0 per defect. presence.T @ presence counts, for each pair, the images where both are 1 (the diagonal = per-class counts from Q3).
Top pair: defect_c & defect_d, 705 images.

In [ ]:
# Q4: co-occurrence of defect types
presence = train.pivot(index="image_id",
                       columns="defect_type",
                       values="has_defect").astype(int)
print(presence.head(3))
print(presence.shape)

In [ ]:
co = presence.T @ presence
print(co)

In [ ]:
cols = list(co.columns)
best_pair, best_count = None, -1

for i in range(len(cols)):
    for j in range(i + 1, len(cols)):          # j > i → only the upper half, skips the diagonal
        if co.iloc[i, j] > best_count:
            best_count = co.iloc[i, j]
            best_pair = (cols[i], cols[j])

print("Top pair:", best_pair, best_count)

## Q5 — Co-occurrence vs correlation
Q4's 705 c&d images look large, but c (36%) and d (15%) are both common: chance alone gives ≈614. Pearson correlation on the 0/1 table removes base rates: c–d = 0.05 (all pairs ≈ 0).
Answer: 0.05, No — the high count is a base-rate effect, not a real association.

In [ ]:
# Q5: correlation vs co-occurrence
corr = presence.corr()          # Pearson correlation between every pair of columns
print(corr.round(2))

print("c-d correlation:", round(corr.loc["defect_c", "defect_d"], 2))

In [ ]:
n = len(presence)
p_c = presence["defect_c"].mean()     # share of images with c (mean of 0/1 = fraction of 1s)
p_d = presence["defect_d"].mean()
print("Expected c&d if independent:", round(n * p_c * p_d, 1), "| Actual:", co.loc["defect_c", "defect_d"])

## Q6 — Pixel-level overlap
For the 1,198 images with 2+ defect types, I decoded all 4 masks and counted pixels that are 1 in both masks, for every pair.
Max overlap = 0 pixels → defects never share a pixel (mutually exclusive), even when they share an image.
Answer: 0, Yes

In [ ]:
# Q6: pixel-level overlap between defect masks
from itertools import combinations

multi_ids = presence.index[presence.sum(axis=1) > 1]
print("Multi-label images:", len(multi_ids))

rle_table = train.pivot(index="image_id", columns="defect_type", values="mask_rle")

In [ ]:
max_overlap = 0

for image_id in multi_ids:
    # decode all 4 masks of this image (empty ones become all-zero)
    masks = {d: rle_decode(rle_table.loc[image_id, d]) for d in rle_table.columns}

    for d1, d2 in combinations(masks, 2):            # every pair of defect types
        overlap = int((masks[d1] & masks[d2]).sum())  # pixels that are 1 in BOTH
        max_overlap = max(max_overlap, overlap)       # remember the biggest seen so far

print("Max overlapping pixels:", max_overlap)

## Q7 — PyTorch Dataset (Paradigm 1)
__getitem__: read image (BGR) → RGB → /255 → (3,128,800) tensor; decode 4 RLEs in fixed class order → stack → (4,128,800) float32.
First batch (batch_size=4): images (4,3,128,800), masks (4,4,128,800), float32.
Masks are float because BCEWithLogitsLoss needs float targets matching the logits' dtype.

In [ ]:
# Q7: PyTorch Dataset (Paradigm 1: one 4-channel mask per image)
import cv2
import torch
from torch.utils.data import Dataset, DataLoader

IMG_DIR = "/kaggle/input/competitions/industrial-surface-defect-detection/public/train"
CLASSES = ["defect_a", "defect_b", "defect_c", "defect_d"]

In [ ]:
class SurfaceAnomalyDataset(Dataset):
    def __init__(self, rle_table, img_dir):
        self.rle_table = rle_table                   # one row per image, 4 RLE columns (from Q6)
        self.image_ids = list(rle_table.index)       # the list of image names
        self.img_dir = img_dir

    def __len__(self):
        return len(self.image_ids)                   # number of unique images

    def __getitem__(self, idx):
        image_id = self.image_ids[idx]

        # --- image ---
        img = cv2.imread(f"{self.img_dir}/{image_id}.jpg")   # (128, 800, 3), BGR, values 0–255
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)           # BGR → RGB
        img = img.astype(np.float32) / 255.0                 # values 0–1
        img = torch.from_numpy(img).permute(2, 0, 1)         # (3, 128, 800)

        # --- masks ---
        masks = [rle_decode(self.rle_table.loc[image_id, c]) for c in CLASSES]  # 4 × (128, 800)
        masks = np.stack(masks, axis=0).astype(np.float32)                      # (4, 128, 800)
        masks = torch.from_numpy(masks)

        return img, masks

In [ ]:
dataset = SurfaceAnomalyDataset(rle_table, IMG_DIR)
print("Dataset length:", len(dataset))

loader = DataLoader(dataset, batch_size=4, shuffle=True)
images, masks = next(iter(loader))

print("images:", images.shape, images.dtype)
print("masks :", masks.shape, masks.dtype)
print("mask values:", torch.unique(masks))

## Q8 — Baseline U-Net training (5 epochs)
Stratified 80/20 split on label combinations. 3-stage U-Net (16→32→64→128 channels, skip connections), 4 logit maps.
BCEWithLogitsLoss, Adam lr 3e-4, batch 8. Train and validation loss logged each epoch; best validation checkpoint saved.
Answer: epoch1 (…, …), epoch5 (…, …), best epoch …, <verdict + 1 sentence>

In [ ]:
# Q8: baseline U-Net, 5 epochs
from sklearn.model_selection import train_test_split

combo = presence.astype(str).agg("".join, axis=1)     # e.g. "0011" = has c and d
train_ids, val_ids = train_test_split(presence.index, test_size=0.2,
                                      random_state=42, stratify=combo)
print("train:", len(train_ids), "| val:", len(val_ids))

train_ds = SurfaceAnomalyDataset(rle_table.loc[train_ids], IMG_DIR)
val_ds   = SurfaceAnomalyDataset(rle_table.loc[val_ids], IMG_DIR)

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=8, shuffle=False, num_workers=2)

In [ ]:
import torch.nn as nn

class DoubleConv(nn.Module):
    """Two 3×3 convolutions, each followed by BatchNorm and ReLU (the basic U-Net block)."""
    def __init__(self, cin, cout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(cin, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
            nn.Conv2d(cout, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.net(x)

class UNet3(nn.Module):
    def __init__(self, in_ch=3, out_ch=4, base=16):
        super().__init__()
        self.pool = nn.MaxPool2d(2)                      # halves height and width
        # encoder (going down)
        self.enc1 = DoubleConv(in_ch, base)              # 16 channels
        self.enc2 = DoubleConv(base, base * 2)           # 32
        self.enc3 = DoubleConv(base * 2, base * 4)       # 64
        self.bottleneck = DoubleConv(base * 4, base * 8) # 128
        # decoder (going up)
        self.up3 = nn.ConvTranspose2d(base * 8, base * 4, 2, stride=2)   # doubles size
        self.dec3 = DoubleConv(base * 8, base * 4)
        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 2, stride=2)
        self.dec2 = DoubleConv(base * 4, base * 2)
        self.up1 = nn.ConvTranspose2d(base * 2, base, 2, stride=2)
        self.dec1 = DoubleConv(base * 2, base)
        self.head = nn.Conv2d(base, out_ch, 1)           # 4 output maps (raw scores)

    def forward(self, x):                                 # x: (B, 3, 128, 800)
        e1 = self.enc1(x)                                 # (B, 16, 128, 800)
        e2 = self.enc2(self.pool(e1))                     # (B, 32,  64, 400)
        e3 = self.enc3(self.pool(e2))                     # (B, 64,  32, 200)
        b  = self.bottleneck(self.pool(e3))               # (B, 128, 16, 100)
        d3 = self.dec3(torch.cat([self.up3(b), e3], 1))   # up + skip → (B, 64,  32, 200)
        d2 = self.dec2(torch.cat([self.up2(d3), e2], 1))  # (B, 32,  64, 400)
        d1 = self.dec1(torch.cat([self.up1(d2), e1], 1))  # (B, 16, 128, 800)
        return self.head(d1)                              # (B, 4,  128, 800)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = UNet3().to(device)
print("Device:", device)
print("Parameters:", sum(p.numel() for p in model.parameters()))

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)

EPOCHS = 5
history = []
best_val = float("inf")

for epoch in range(1, EPOCHS + 1):
    # ---- train ----
    model.train()
    train_loss = 0.0
    for images, masks in train_loader:
        images, masks = images.to(device), masks.to(device)
        optimizer.zero_grad()                   # clear old gradients
        logits = model(images)                  # 1. predict
        loss = criterion(logits, masks)         # 2. measure error
        loss.backward()                         # 3. work out how to adjust each weight
        optimizer.step()                        # 4. adjust them a little
        train_loss += loss.item() * images.size(0)
    train_loss /= len(train_ds)

    # ---- validate ----
    model.eval()
    val_loss = 0.0
    with torch.no_grad():                       # no learning here, just measuring
        for images, masks in val_loader:
            images, masks = images.to(device), masks.to(device)
            val_loss += criterion(model(images), masks).item() * images.size(0)
    val_loss /= len(val_ds)

    history.append((epoch, train_loss, val_loss))
    print(f"Epoch {epoch}: train {train_loss:.4f} | val {val_loss:.4f}")

    if val_loss < best_val:                     # keep the best checkpoint
        best_val = val_loss
        torch.save(model.state_dict(), "unet3_best.pth")
        print("  saved best model")

In [ ]:
best_epoch = min(history, key=lambda h: h[2])[0]
e1, e5 = history[0], history[-1]
print(f"epoch1 (train {e1[1]:.4f}, val {e1[2]:.4f}) | epoch5 (train {e5[1]:.4f}, val {e5[2]:.4f}) | best epoch {best_epoch}")